### `create wind ninja input...`

#### moving this into the other repo `snow_model_forcing` to get a clean repo going there

Notebook contents 
* dropping code here which I will use to create wind ninja input data files

created by Cassie Lumbrazo\
last updated: July 2025\
run location: UAS linux\
python environment: **rasterio**

In [2]:
# import packages 
%matplotlib inline

# plotting packages 
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns 

sns.set_theme()
# plt.rcParams['figure.figsize'] = [12,6] #overriding size

# data packages 
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime

import scipy

✅ Step 1: Convert NetCDF to .asc files per timestep

Each .asc will need a header like before, 

I’ll write a function that:

Reads each timestep of wind_speed and wind_direction.

Saves them to .asc files with the timestamp in the filename.

Returns a list of generated filenames.

✅ Step 2: Run WindNinja CLI for each timestep

We'll write a script to:

Copy the base config.

Update the input_wind_filename in the config to the .asc just made.

Run WindNinja using subprocess.

✅ Step 3: Zip results per batch

After WindNinja finishes, we zip the result directories.

### Dropping code for now, come back to this

Here's your pipeline script that:

Extracts wind data from a NetCDF timeseries.

Converts each timestep into .asc files for WindNinja.

Runs WindNinja for each timestep using a config template.

Zips all output folders into one archive.

In [ ]:
import os
import xarray as xr
import numpy as np
import subprocess
from zipfile import ZipFile
from datetime import datetime

# ==== USER SETTINGS (EDIT HERE) ====
NETCDF_FILE = "path/to/your/wind_data.nc"
DEM_FILE = "path/to/your/input_dem.asc"
WINDNINJA_CONFIG = "path/to/your/windninja_config.cfg"
OUTPUT_DIR = "windninja_output"
TMP_ASC_DIR = "ascii_inputs"
VARIABLES = {
    "speed": "wind_speed",
    "direction": "wind_direction"
}
XLLCORNER = 2782000  # change as needed
YLLCORNER = 1182000  # change as needed
CELLSIZE = 100
NODATA_VALUE = -9999
# ===================================

def write_asc(data: np.ndarray, output_path: str, xll: float, yll: float, cellsize: float, nodata: float):
    nrows, ncols = data.shape
    header = (
        f"ncols         {ncols}\n"
        f"nrows         {nrows}\n"
        f"xllcorner     {xll}\n"
        f"yllcorner     {yll}\n"
        f"cellsize      {cellsize}\n"
        f"NODATA_value  {nodata}\n"
    )
    with open(output_path, 'w') as f:
        f.write(header)
        for row in data:
            row_str = '\t'.join(f"{val:.2f}" if not np.isnan(val) else str(nodata) for val in row)
            f.write(row_str + '\n')

def generate_ascii_files(ds):
    os.makedirs(TMP_ASC_DIR, exist_ok=True)
    ascii_file_pairs = []
    for i, t in enumerate(ds.time):
        timestamp = pd.to_datetime(str(t.values)).strftime("%Y%m%d_%H%M")
        speed_2d = ds[VARIABLES["speed"]].isel(time=i).values
        dir_2d = ds[VARIABLES["direction"]].isel(time=i).values

        speed_path = os.path.join(TMP_ASC_DIR, f"wind_speed_{timestamp}.asc")
        dir_path = os.path.join(TMP_ASC_DIR, f"wind_dir_{timestamp}.asc")

        write_asc(speed_2d, speed_path, XLLCORNER, YLLCORNER, CELLSIZE, NODATA_VALUE)
        write_asc(dir_2d, dir_path, XLLCORNER, YLLCORNER, CELLSIZE, NODATA_VALUE)

        ascii_file_pairs.append((timestamp, speed_path, dir_path))
    return ascii_file_pairs

def run_windninja(speed_path, dir_path, timestamp):
    output_subdir = os.path.join(OUTPUT_DIR, f"windninja_{timestamp}")
    os.makedirs(output_subdir, exist_ok=True)

    config_path = os.path.join(output_subdir, f"config_{timestamp}.cfg")
    with open(WINDNINJA_CONFIG, 'r') as f:
        config = f.read()

    config = config.replace("INPUT_SPEED_FILE", speed_path)
    config = config.replace("INPUT_DIR_FILE", dir_path)
    config = config.replace("INPUT_DEM_FILE", DEM_FILE)
    config = config.replace("OUTPUT_PATH", output_subdir)

    with open(config_path, 'w') as f:
        f.write(config)

    subprocess.run(["WindNinja_cli", config_path], check=True)

def zip_outputs():
    zipf = ZipFile(f"{OUTPUT_DIR}.zip", 'w')
    for root, _, files in os.walk(OUTPUT_DIR):
        for file in files:
            filepath = os.path.join(root, file)
            zipf.write(filepath, os.path.relpath(filepath, OUTPUT_DIR))
    zipf.close()

def main():
    ds = xr.open_dataset(NETCDF_FILE)
    ascii_files = generate_ascii_files(ds)
    for timestamp, speed, direction in ascii_files:
        print(f"Processing timestep {timestamp}")
        run_windninja(speed, direction, timestamp)
    zip_outputs()
    print("All done. Output zipped.")

if __name__ == "__main__":
    import pandas as pd
    main()
